In [1]:
import torch

from langchain_docling import DoclingLoader
FILE_PATH = [r"C:\Users\Archi\Desktop\student_assistant\data\BTech_FAQs.pdf"]  # Docling Technical Report
loader = DoclingLoader(file_path=FILE_PATH)
docs = loader.load()

C:\Users\Archi\Desktop\student_assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
No OCR engine found. Please review the install details.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 1298.97it/s]
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


In [2]:
for d in docs:
    print(d.page_content)

FREQUENTLY ASKED QUESTIONS (FAQs)
Institute of Engineering & Technology (IET), DAVV, Indore
Q1. What is the status and recognition of IET DAVV?
IET DAVV is a premier government engineering institute established in 1996 under Devi Ahilya Vishwavidyalaya (DAVV), a State University of Madhya Pradesh.
The institute offers AICTE-approved undergraduate, postgraduate, and doctoral programmes in engineering, technology, management, and applied sciences.
Q2. Where is IET DAVV located?
The institute is located at:
Institute of Engineering & Technology (IET) Vikramshila Campus, Khandwa Road, Indore - 452017, Madhya Pradesh, India.
Indore is one of India's leading educational and industrial cities, offering excellent academic and career opportunities.
Q3. How can I get admission to B.Tech. programmes at IET DAVV?
Admission to B.Tech. programmes is carried out through the counselling process conducted by the Directorate of Technical Education (DTE), Bhopal, Madhya Pradesh, based on JEE (Main) merit

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
data = "".join(doc.page_content for doc in docs)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
texts = text_splitter.split_text(data)

In [4]:
print(texts[1])

The institute offers AICTE-approved undergraduate, postgraduate, and doctoral programmes in engineering, technology, management, and applied sciences.Q2. Where is IET DAVV located?
The institute is located at:


In [9]:
print(data)

FREQUENTLY ASKED QUESTIONS (FAQs)
Institute of Engineering & Technology (IET), DAVV, IndoreQ1. What is the status and recognition of IET DAVV?
IET DAVV is a premier government engineering institute established in 1996 under Devi Ahilya Vishwavidyalaya (DAVV), a State University of Madhya Pradesh.
The institute offers AICTE-approved undergraduate, postgraduate, and doctoral programmes in engineering, technology, management, and applied sciences.Q2. Where is IET DAVV located?
The institute is located at:
Institute of Engineering & Technology (IET) Vikramshila Campus, Khandwa Road, Indore - 452017, Madhya Pradesh, India.
Indore is one of India's leading educational and industrial cities, offering excellent academic and career opportunities.Q3. How can I get admission to B.Tech. programmes at IET DAVV?
Admission to B.Tech. programmes is carried out through the counselling process conducted by the Directorate of Technical Education (DTE), Bhopal, Madhya Pradesh, based on JEE (Main) merit/ra

In [5]:
pip install -qU langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Embed multiple documents
vectors = embeddings.embed_documents([data])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1318.39it/s]


In [7]:
!pip install sentence_transformers

In [14]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [13]:
from langchain_community.vectorstores.utils import filter_complex_metadata

docs = filter_complex_metadata(docs)
fr

C:\Users\Archi\AppData\Local\Temp\ipykernel_11284\3825989241.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores.utils import filter_complex_metadata


In [12]:
!pip install langchain-community


  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached langchain_classic-1.0.8-py3-none-any.whl.metadata (5.1 kB)
Using cached langchain_community-0.4.2-py3-none-any.whl (2.4 MB)
Using cached httpx_sse-0.4.3-py3-none-any.whl (9.0 kB)
Using cached langchain_classic-1.0.8-py3-none-any.whl (1.0 MB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ------------- -------------------------- 0.8/2.4 MB 4.4 MB/s eta 0:00:01
   -------------------------- ------------- 1.6/2.4 MB 4.2 MB/s eta 0:00:01
   ---------------------------------- ----- 2.1/2.4 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 3.1 MB/s  0:00:00

   ---------------------------------------- 0/4 [sqlalchemy]
   ---------------------------------------- 0/4 [sqlalchemy]
   ---------------------------------------- 0/4 [sqlalchemy]
   ---------------------------------------- 0/4

In [ ]:
!pip install langchain-chroma


In [33]:
def ask_rag(question):
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)
    prompt = f"""
    you are student assitant you have info about iet college in your context give answers accordingly
    be specific with answers
    Context:
    {context}

    Question:
    {question}
    """
    response = model.invoke(prompt)
    return response.content


In [36]:
input = "what are the programs in iet davv"
ask_rag(input)

'**Programs offered at IET\u202fDAVV (Devi Ahilya Vishwavidyalaya)**  \n\n| Level | Discipline | Specific Programs |\n|-------|------------|-------------------|\n| **Under‑Graduate (B.Tech.)** | Core & Emerging Engineering | • B.Tech. – Computer Science & Engineering (CSE)  <br>• B.Tech. – Electronics & Communication Engineering (ECE)  <br>• B.Tech. – Mechanical Engineering (ME)  <br>• B.Tech. – Civil Engineering (CE)  <br>• **B.Tech. – Industrial Production (IP)** *(newly introduced)*  <br>• **B.Tech. – Electrical & Electronics Engineering (EEE)** *(newly introduced)*  <br>• **B.Tech. – Computer Science & Business Systems (CSBS)** *(newly introduced)* |\n| **Post‑Graduate (M.Tech.)** | Specialized Engineering | • M.Tech. – Computer Science & Engineering  <br>• M.Tech. – Electronics & Communication Engineering  <br>• M.Tech. – Mechanical Engineering  <br>• **M.Tech. – Microelectronics & VLSI Design** *(newly introduced)* |\n| **Management** | Business & Technology | • MBA – General Man

In [35]:
import os
model = ChatGroq(api_key=os.getenv("GROQ_API_KEY")
                 ,temperature=0.0,
                 model = "openai/gpt-oss-120b")

In [25]:
from dotenv import load_dotenv
load_dotenv()

True

In [27]:
from langchain_groq import ChatGroq